# Bài tập Perceptron – Quyết định tưới cây

**Tình huống:** quyết định tự động bật/tắt máy bơm tưới cây.

### Nhãn
- `Relay = 0`: Chưa cần tưới
- `Relay = 1`: Cần tưới

### Feature
- `Moist`: độ ẩm đất
- `Temperature`: nhiệt độ không khí
- `HoursSinceWatering`: số giờ từ lần tưới gần nhất
- `RainChance`: khả năng mưa

Quy trình:
1. Đọc dữ liệu CSV.
2. Kiểm tra dữ liệu.
3. Chọn feature và nhãn.
4. Chia train/test.
5. Chuẩn hóa dữ liệu.
6. Huấn luyện Perceptron.
7. Tính `net`, bias và trọng số.
8. Đánh giá Accuracy, Classification Report, Confusion Matrix.
9. Xuất kết quả test.
10. Dự đoán một trường hợp mới.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# Housing.csv trước đây được thay hoàn toàn bằng dữ liệu tưới cây.
# CSV nằm cùng thư mục với notebook.

BASE_DIR = Path.cwd()
CSV_FILE = BASE_DIR / "du_lieu_tuoi_cay_perceptron(2).csv"

print("Đường dẫn CSV:")
print(CSV_FILE)

if not CSV_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file CSV tại: {CSV_FILE}\n"
        "Hãy đặt CSV cùng thư mục notebook."
    )

df = pd.read_csv(CSV_FILE)

print("\nĐọc dữ liệu thành công!")
print("Kích thước:", df.shape)
display(df.head())


In [ ]:
# Kiểm tra toàn bộ dữ liệu
print("Các cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu:")
print(df.dtypes)

print("\nSố giá trị thiếu:")
print(df.isnull().sum())

print("\nPhân bố Relay:")
print(df["Relay"].value_counts().sort_index())


In [ ]:
# Đặt feature và target

FEATURE_COLUMNS = [
    "Moist",
    "Temperature",
    "HoursSinceWatering",
    "RainChance",
]

TARGET_COLUMN = "Relay"

CLASS_NAMES = {
    0: "Chưa cần tưới",
    1: "Cần tưới",
}

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("Feature:")
for feature in FEATURE_COLUMNS:
    print("-", feature)

print("\nTarget:", TARGET_COLUMN)
print("\nSố mẫu mỗi lớp:")
print(y.map(CLASS_NAMES).value_counts())


## Ý nghĩa các feature

- **Moist:** độ ẩm đất càng thấp thì cây thường có xu hướng cần tưới.
- **Temperature:** nhiệt độ cao có thể làm tăng nhu cầu nước.
- **HoursSinceWatering:** càng lâu kể từ lần tưới gần nhất thì nhu cầu tưới có thể tăng.
- **RainChance:** khả năng mưa cao có thể làm giảm nhu cầu bật máy bơm.

Các feature được đưa vào Perceptron **đồng thời**, không xây dựng một quy tắc đơn duy nhất.


In [ ]:
# Chia dữ liệu train/test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


In [ ]:
# Pipeline:
# StandardScaler -> Perceptron

model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "perceptron",
        Perceptron(
            max_iter=1000,
            tol=1e-3,
            eta0=0.1,
            random_state=42,
            shuffle=True,
        ),
    ),
])

model.fit(X_train, y_train)

print("Huấn luyện Perceptron hoàn tất.")


In [ ]:
# Dự đoán và đánh giá

y_pred = model.predict(X_test)
net_scores = model.decision_function(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=[
            CLASS_NAMES[0],
            CLASS_NAMES[1],
        ],
        zero_division=0,
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


In [ ]:
# Trọng số, bias và net

perceptron = model.named_steps["perceptron"]

print("Bias b:", round(float(perceptron.intercept_[0]), 6))
print("Số vòng lặp:", perceptron.n_iter_)

print("\nTrọng số:")
for feature, weight in zip(
    FEATURE_COLUMNS,
    perceptron.coef_[0],
):
    print(f"{feature}: {weight:.6f}")

print("\n5 giá trị net đầu tiên:")
print(np.round(net_scores[:5], 6))


In [ ]:
# Bảng kết quả tập test

test_result = df.loc[X_test.index].copy()
test_result["nhan_that"] = y_test
test_result["du_doan"] = y_pred
test_result["net"] = net_scores
test_result["ket_qua"] = np.where(
    test_result["nhan_that"] == test_result["du_doan"],
    "Đúng",
    "Sai",
)

display(test_result)

print(
    "Số mẫu dự đoán sai:",
    (test_result["ket_qua"] == "Sai").sum()
)


In [ ]:
# Xuất kết quả test

output_file = BASE_DIR / "ket_qua_test_tuoi_cay.csv"

test_result.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
)

print("Đã tạo:", output_file)


## Dự đoán trường hợp mới

Nhập 4 thông số:
- độ ẩm đất
- nhiệt độ
- số giờ từ lần tưới gần nhất
- khả năng mưa


In [ ]:
sample = {
    "Moist": 35,
    "Temperature": 32,
    "HoursSinceWatering": 8,
    "RainChance": 15,
}

new_data = pd.DataFrame(
    [sample],
    columns=FEATURE_COLUMNS,
)

prediction = int(model.predict(new_data)[0])
net = float(model.decision_function(new_data)[0])

display(new_data)

print("net =", round(net, 6))
print("Relay =", prediction)
print("Quyết định =", CLASS_NAMES[prediction])


## Kết luận

Mô hình Perceptron sử dụng đồng thời 4 feature `Moist`, `Temperature`, `HoursSinceWatering` và `RainChance` để đưa ra quyết định `Relay`.

- `0`: chưa cần tưới.
- `1`: cần tưới.

Dữ liệu được chuẩn hóa trước khi đưa vào Perceptron. Kết quả được đánh giá bằng Accuracy, Classification Report và Confusion Matrix.
